# Chess Expert — Train & Play on Colab

Downloads grandmaster games, trains the `ChessPolicyNet` on a GPU, then lets you **play against it on a clickable board right here in Colab**.

**Before you run anything:**
1. `Runtime → Change runtime type →` **GPU**. A **T4** works but is slow; an **L4** (best value) or **A100** is much faster and enables mixed precision. High-RAM if offered.
2. In the *Clone* cell, set `REPO_URL` to your GitHub repo.

> 💡 **Dry run first.** Before a long paid run, run the whole notebook once with `--max-games 200` (Parse cell) and `--epochs 2` (Train cell) to prove the chain works for pennies. Then remove the caps.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 2. Get the code and install dependencies
Set `REPO_URL` to your repo (after you push it to GitHub).

In [ ]:
REPO_URL = "https://github.com/AhPro7/chess-expert.git"  # <-- your repo

import os
if not os.path.isdir("chess-expert"):
    !git clone $REPO_URL
%cd chess-expert
!git pull
!pip -q install -r requirements.txt

## 3. Download grandmaster games
Real GM archives (Carlsen, Kasparov, Fischer, Karpov, Anand, ...) merged into `data/gm_games.pgn`. Edit `scripts/download_data.sh` to add players.

In [ ]:
!bash scripts/download_data.sh
!ls -lh data/gm_games.pgn

## 4. Parse PGN → training samples
Stored as uint8 + memory-mapped, so RAM stays flat. Keep all GM games (`--min-elo 0`). For a quick pass add `--max-games 5000`.

In [ ]:
!python -m src.data --pgn data/gm_games.pgn --out data/samples --min-elo 0

## 5. Train
TF32 + cuDNN autotune are always on (big, safe speedup on L4/A100). On those cards also try `--amp on` for bf16 mixed precision. Watch **val move-match**; it usually plateaus by ~10–15 epochs, so you can stop early.

In [ ]:
!python -m src.train \
  --data data/samples \
  --out models/chess_expert.pt \
  --epochs 20 \
  --batch-size 4096 \
  --lr 1e-3 \
  --channels 128 --blocks 10
# On L4/A100 add  --amp on  for extra speed. OOM? lower --batch-size (e.g. 2048).

## 6. Sanity check — self-play a full game (asserts every move is legal)

In [ ]:
!python -m src.play --checkpoint models/chess_expert.pt --plies 60

## 7. 🎮 Play against it — clickable board
Run the cell, then **click one of your pieces and click where it should go**. The engine replies. **New game** restarts. Play Black with `human_white=False`; add variety with `temperature=0.3`.

In [ ]:
from demo.colab_gui import play
play("models/chess_expert.pt")  # you are White

## 8. Make the demo GIF (for LinkedIn) and download it

In [ ]:
!python -m demo.make_gif --checkpoint models/chess_expert.pt --out demo/self_play.gif --plies 60 --temperature 0.6
from google.colab import files
files.download('demo/self_play.gif')

## 9. Download the trained model
Save `chess_expert.pt` into your local repo's `models/` folder (and commit it so anyone can play on clone).

In [ ]:
from google.colab import files
files.download('models/chess_expert.pt')